In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
import lightgbm as lgb
from catboost import CatBoostClassifier
from catboost import CatBoostRegressor
from collections import Counter
from sklearn.metrics import mean_squared_error
from pathlib import Path

project_root = Path.cwd().parent

op = pd.read_excel(project_root/'L3'/'Sales_Prediction_with_Probabilities_&_SCORES.xlsx')

In [2]:

clean_data=['Shape','Size ranges','Color-ranges','Clarity-ranges','Lab','Stone Location','Fluorescence','Cut','Polish']
for col in clean_data:
    op[col]=op[col].fillna("Unknown")

op_model = op[op['Prediction'] != "No Sales data"].copy()


#MODEL 1

In [3]:
features=['Shape','Size ranges','Color-ranges','Clarity-ranges','Lab','Stone Location','Fluorescence','Cut','Polish','Ratio']
X=op_model[features]
y=op_model['Prediction']

counts_1 = Counter(y)
total_1 = sum(counts_1.values())

class_weights_dict = {
    cls: total_1 / count for cls, count in counts_1.items()
}


X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model_loc=CatBoostClassifier(
    iterations=600,
    depth=7,
    learning_rate=0.05,
    loss_function='MultiClass',
    random_seed=42,
    class_weights=class_weights_dict,
    verbose=0
)

model_path_1 = project_root/'L3'/'models'/"model_loc.pkl"

if os.path.exists(model_path_1):
    print("Loading Model 1...")
    model_loc = joblib.load(model_path_1)
else:
    print("Training Model 1...")
    model_loc.fit(X_train, y_train,
                  cat_features=['Shape','Size ranges','Color-ranges','Clarity-ranges','Lab','Stone Location','Fluorescence','Cut','Polish'])
    joblib.dump(model_loc, model_path_1)


y_pred=model_loc.predict(X_test).ravel()

Loading Model 1...


#Acuuracy Score


In [4]:
from sklearn.metrics import accuracy_score
print("Accuracy:",accuracy_score(y_test,y_pred))

op['Loc_pred_ML'] = model_loc.predict(op[features]).ravel()

probs = model_loc.predict_proba(op[features])
probs_sorted = np.sort(probs, axis=1)

op['Confidence'] = probs_sorted[:, -1] - probs_sorted[:, -2]

threshold = 0.10

op['ML_Sales_Pred'] = op.apply(
    lambda row: row['Loc_pred_ML'] if row['Confidence'] >= threshold else "Not Sure",
    axis=1
)


Accuracy: 0.996822033898305


In [5]:
print(op_model['Prediction'].value_counts(normalize=True))

Prediction
Sell to HK     0.948294
Sell to USA    0.051706
Name: proportion, dtype: float64


In [6]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

  Sell to HK       1.00      1.00      1.00       895
 Sell to USA       0.94      1.00      0.97        49

    accuracy                           1.00       944
   macro avg       0.97      1.00      0.98       944
weighted avg       1.00      1.00      1.00       944



#MODEL 2 for Sales Label Prediction

In [7]:
label_feat=['Shape','Color-ranges','Clarity-ranges','Size ranges','Lab','Stone Location','Fluorescence','Cut','Polish','Ratio']
X_2=op_model[label_feat]
y_2=op_model["Sales Label"]

from collections import Counter

counts_2 = Counter(y_2)
total_2 = sum(counts_2.values())

class_weights_2 = {
    cls: total_2/count for cls, count in counts_2.items()
}

X_2_train,X_2_test,y_2_train,y_2_test=train_test_split(
    X_2,y_2,
    test_size=0.2,
    random_state=42,
    stratify=y_2,
    
)

model_label=CatBoostClassifier(
    iterations=600,
    depth=7,
    learning_rate=0.05,
    loss_function='MultiClass',
    class_weights=class_weights_2,
    verbose=0
)

model_path_2 = project_root/'L3'/'models'/"model_label.pkl"

if os.path.exists(model_path_2):
    print("Loading Model 2...")
    model_label = joblib.load(model_path_2)
else:
    print("Training Model 2...")
    model_label.fit(X_2_train, y_2_train,
                    cat_features=['Shape','Color-ranges','Clarity-ranges','Size ranges','Lab','Stone Location','Fluorescence','Cut','Polish'])
    joblib.dump(model_label, model_path_2)


y_2_pred=model_label.predict(X_2_test).ravel()

Loading Model 2...


In [8]:
op['Label_pred'] = model_label.predict(op[label_feat]).ravel()

#MODEL 3 for Sales Probability


In [9]:
features_3 = [
    'Shape','Color-ranges','Clarity-ranges','Size ranges','Lab',
    'Stone Location','Fluorescence','Cut','Polish','Ratio'
]

X_3 = op_model[features_3]
y_3 = op_model['Sales_Probability'] 

X_3_train,X_3_test,y_3_train,y_3_test=train_test_split(
    X_3,y_3,
    test_size=0.2,
    random_state=42,
    
)

from catboost import CatBoostRegressor

model_prob = CatBoostRegressor(
    iterations=600,
    depth=7,
    learning_rate=0.05,
    loss_function='RMSE',
    random_seed=42,
    verbose=0
)

model_path_3 = project_root/'L3'/'models'/"model_prob.pkl"

if os.path.exists(model_path_3):
    print("Loading Model 3...")
    model_prob = joblib.load(model_path_3)
else:
    print("Training Model 3...")
    model_prob.fit(X_3_train, y_3_train,
                   cat_features=['Shape','Color-ranges','Clarity-ranges','Size ranges','Lab','Stone Location','Fluorescence','Cut','Polish'])
    joblib.dump(model_prob, model_path_3)

Loading Model 3...


In [10]:
y_3_pred=model_prob.predict(X_3_test).ravel()

op['Sales_Prob_ML'] = model_prob.predict(op[features_3])
op['Sales_Prob_ML'] = op['Sales_Prob_ML'].clip(0,1)



prob_conditions=[
    op['Sales_Prob_ML']>=0.75,  
    (op['Sales_Prob_ML']>=0.40) & (op['Sales_Prob_ML']<0.75),
    (op['Sales_Prob_ML']<0.40)
]

prob_choices=[
    "Fast Moving",
    "Slow Moving",
    "Dead Stock"
]
op['Sales_Prob_label_ML']=np.select(prob_conditions, prob_choices,default='')

#MODEL4 FOR Prob(0-30)	Prob(30-90)	Prob(90-150)	Prob(150-365)	Prob(365+)


In [11]:
target_cols_prob = [
    'Prob(0-30)',
    'Prob(30-90)',
    'Prob(90-150)',
    'Prob(150-365)',
    'Prob(365+)'
]

features_4 = ['Shape','Color-ranges','Clarity-ranges','Size ranges','Lab',
              'Stone Location','Fluorescence','Cut','Polish',
              'Ratio']

X_4 = op_model[features_4]


In [12]:
models = {}

for col in target_cols_prob:
    
    safe_col = col.replace("(", "").replace(")", "").replace("+", "plus").replace("-", "_")
    model_path = project_root/'L3'/'models'/f"model_{safe_col}.pkl"

    if os.path.exists(model_path):
        print(f"Loading Model for {col}...")
        model_data = joblib.load(model_path)
    else:
        print(f"Training Model for {col}...")
        
        y_4 = op_model[col]

        X_4_train, X_4_test, y_4_train, y_4_test = train_test_split(
            X_4, y_4,
            test_size=0.2,
            random_state=42
        )

        model_data = CatBoostRegressor(
            iterations=600,
            depth=6,
            learning_rate=0.05,
            loss_function='RMSE',
            verbose=0
        )

        model_data.fit(
            X_4_train, y_4_train,
            cat_features=['Shape','Color-ranges','Clarity-ranges','Size ranges','Lab',
                          'Stone Location','Fluorescence','Cut','Polish']
        )

        joblib.dump(model_data, model_path)

    # Store model
    models[col] = model_data

    # Predict
    op[f'ML_{col}'] = model_data.predict(op[features_4]).clip(0,1)


# Normalize probabilities
prob_cols_ml = [f'ML_{col}' for col in target_cols_prob]

row_sums = op[prob_cols_ml].sum(axis=1)

op[prob_cols_ml] = op[prob_cols_ml].div(
    row_sums.replace(0, 1), axis=0
) 
    
  
    
    

Loading Model for Prob(0-30)...


Loading Model for Prob(30-90)...


Loading Model for Prob(90-150)...
Loading Model for Prob(150-365)...


Loading Model for Prob(365+)...


#MODEL 5 For AGING RISK SCORE

In [13]:

features_5 = ['Shape','Color-ranges','Clarity-ranges','Size ranges','Lab',
              'Stone Location','Fluorescence','Cut','Polish',
              'Ratio','Inventory Age','Certification Age']

X_5=op_model[features_5]
y_5=op_model['Aging Risk Score']

In [14]:
model_age_risk = CatBoostRegressor(
        iterations=600,
        depth=6,
        learning_rate=0.05,
        loss_function='RMSE',
        verbose=0
    )

X_5_train,X_5_test,y_5_train,y_5_test=train_test_split(
    X_5,y_5,
    test_size=0.2,
    random_state=42,
 )

model_path_5 = project_root/'L3'/'models'/"model_age_risk.pkl"

if os.path.exists(model_path_5):
    print("Loading Model 5...")
    model_age_risk = joblib.load(model_path_5)
else:
    print("Training Model 5...")
    model_age_risk.fit(
        X_5_train, y_5_train,
        cat_features=['Shape','Color-ranges','Clarity-ranges','Size ranges','Lab','Stone Location','Fluorescence','Cut','Polish']
    )
    joblib.dump(model_age_risk, model_path_5)

y_5_pred=model_age_risk.predict(X_5_test)
op['ML_Aging_Risk_Score']=model_age_risk.predict(op[features_5]).clip(0,100)

Loading Model 5...


In [15]:
conditions = [
    op['ML_Aging_Risk_Score'] >= 75,
    (op['ML_Aging_Risk_Score'] >= 45) & (op['ML_Aging_Risk_Score'] < 75),
    op['ML_Aging_Risk_Score'] < 45
]

choices = ["High", "Medium", "Low"]

op['Risk Classification'] = np.select(conditions, choices, default='')

#Features

In [16]:
window_midpoints = {
    'Prob(0-30)': 15,
    'Prob(30-90)': 60,
    'Prob(90-150)': 120,
    'Prob(150-365)': 250,
    'Prob(365+)': 400
}

op['Expected_Days_to_sell'] = sum(
    op[f'ML_{k}'] * v for k, v in window_midpoints.items()
)






#POSITION ALLOCATION

In [17]:
pos_loc=op.columns.get_loc('Prediction')
data_ml_loc=op.pop('Loc_pred_ML')
final_data=op.pop('ML_Sales_Pred')
op.insert(pos_loc+1,'Loc_pred_ML',data_ml_loc)
op.insert(pos_loc+2,'ML_Sales_Pred',final_data)


pos_label=op.columns.get_loc('Sales Label')
data_label=op.pop('Label_pred')
op.insert(pos_label+1,'Label_pred',data_label)


pos_prob=op.columns.get_loc('Sales_Probability')
data_prob=op.pop('Sales_Prob_ML')
data_prob_label=op.pop('Sales_Prob_label_ML')
op.insert(pos_prob+1,'ML_Sales_Prob',data_prob)
op.insert(pos_prob+2,'Stock_Classification',data_prob_label)

pos_data=op.columns.get_loc('Prob(365+)')
data_1=op.pop('ML_Prob(0-30)')
data_2=op.pop('ML_Prob(30-90)')
data_3=op.pop('ML_Prob(90-150)')
data_4=op.pop('ML_Prob(150-365)')
data_5=op.pop('ML_Prob(365+)')
op.insert(pos_data+1,'Probability_Sell_(0-30)_Days',data_1)
op.insert(pos_data+2,'Probability_Sell_(30-90)_Days',data_2)
op.insert(pos_data+3,'Probability_Sell_(90-150)_Days',data_3)
op.insert(pos_data+4,'Probability_Sell_(150-365)_Days',data_4)
op.insert(pos_data+5,'Probability_Sell_(365+)_Days',data_5)


pos_age_risk=op.columns.get_loc('Aging Risk Score')
data_asr=op.pop('ML_Aging_Risk_Score')
op.insert(pos_age_risk+1,'ML_Aging_Risk_Score',data_asr)

pos_exp=op.columns.get_loc('Probability_Sell_(365+)_Days')
data_exp=op.pop('Expected_Days_to_sell')
op.insert(pos_exp+1,'Expected_Days_to_sell',data_exp)

pos_sales=op.columns.get_loc('Total Sales')
data_sales=op.pop('Sales in HK')
op.insert(pos_sales+1,'Sales in HK',data_sales)


				

#Type Changing /Rounding Off

In [18]:
cols_round=['Prob(0-30)',
            'Prob(30-90)',
            'Prob(90-150)',
            'Prob(150-365)',
            'Prob(365+)',
            'Probability_Sell_(0-30)_Days',	
            'Probability_Sell_(30-90)_Days',	
            'Probability_Sell_(90-150)_Days',	
            'Probability_Sell_(150-365)_Days',	
            'Probability_Sell_(365+)_Days',
            'Sales_Probability',	
            'ML_Sales_Prob',
            'Expected_Days_to_sell',

            ]

for cols in cols_round:
    op[cols]=op[cols].round(2)

op['ML_Aging_Risk_Score']=op['ML_Aging_Risk_Score'].round(0)
op['Expected_Days_to_sell']=op['Expected_Days_to_sell'].round(0)

#RMSE MODEL 3

In [19]:
rmse = np.sqrt(mean_squared_error(y_3_test, y_3_pred))
print("Model 3 RMSE:", rmse)

Model 3 RMSE: 0.047435291825683644


#RMSE MODEL 5

In [20]:
rmse = np.sqrt(mean_squared_error(y_5_test, y_5_pred))
print("Model 5 RMSE:", rmse)

Model 5 RMSE: 5.339427324232051


#DROPPING COLS

In [21]:
op=op.drop(columns=['Loc_pred_ML','Confidence','Sales_Probability_label'])

#OUTPUT FILE


In [22]:
op.to_excel(project_root/'L3'/'ML_Prediction_Level3.xlsx',index=False)
